In [1]:
from typing import Annotated, TypedDict

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import AnyMessage, AIMessage, BaseMessage

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command
from dotenv import load_dotenv

load_dotenv()


True

In [2]:
llm=ChatGoogleGenerativeAI(model="gemini-3.5-flash")

In [3]:
from langgraph.graph.message import add_messages

class ChatState(TypedDict):

    messages:Annotated[list[BaseMessage],add_messages]

In [5]:
def chat_node(state:ChatState):
    decision = interrupt({
        "type":"approval",
        "reason":"Model is about to answer the user question",
        "question":state["messages"][-1].content,
        "instruction":"Approve this question?yes/no"
        })

    if decision["approved"] == 'no':
        return {"messages": [AIMessage(content="Not approved.")]}

    else:
        response = llm.invoke(state["messages"])
        return {"messages": [response]}


In [ ]:
builder = StateGraph(ChatState)
builder.add_node("chat",chat_node)
builder.add_edge(START,"chat")
builder.add_edge("chat",END)

#checkpointer is required for interrupts-helps in pausing flow and saving state
checkpointer=MemorySaver()
app=builder.compile(checkpointer=checkpointer)

In [9]:
# Create a new thread id for this conversation
config = {"configurable": {"thread_id": '1234'}}

# ----- STEP 1: user asks a question -----
initial_input = {
    "messages": [
        ("user", "Explain gradient descent in very simple terms.")
    ]
}

# Invoke the graph for the first time
result = app.invoke(initial_input, config=config)


In [10]:
result

{'messages': [HumanMessage(content='Explain gradient descent in very simple terms.', additional_kwargs={}, response_metadata={}, id='5b9c5443-edc8-4458-bbf6-eaf6d500dc0e')],
 '__interrupt__': [Interrupt(value={'type': 'approval', 'reason': 'Model is about to answer the user question', 'question': 'Explain gradient descent in very simple terms.', 'instruction': 'Approve this question?yes/no'}, id='37b514f822d043f4aba84392ca896f36')]}

In [11]:
message=result['__interrupt__'][0].value
message

{'type': 'approval',
 'reason': 'Model is about to answer the user question',
 'question': 'Explain gradient descent in very simple terms.',
 'instruction': 'Approve this question?yes/no'}

In [ ]:
user_input=input(f"\nBackend message - {message} \n Approve this question?(y/n): ")

<>:1: SyntaxWarning: invalid escape sequence '\ '
<>:1: SyntaxWarning: invalid escape sequence '\ '
C:\Users\pooji\AppData\Local\Temp\ipykernel_20524\1522864935.py:1: SyntaxWarning: invalid escape sequence '\ '
  user_input=input(f"\nBackend message - {message} \ n Approve this question?(y/n): ")


In [13]:
final_result=app.invoke(Command(resume={"approved":user_input}),
                        config=config)

In [14]:
print(final_result)

{'messages': [HumanMessage(content='Explain gradient descent in very simple terms.', additional_kwargs={}, response_metadata={}, id='5b9c5443-edc8-4458-bbf6-eaf6d500dc0e'), AIMessage(content=[{'type': 'text', 'text': 'Imagine you are blindfolded and placed somewhere on a foggy mountain. Your goal is to find the very bottom of the valley. \n\nSince you can’t see, how do you get there?\n\n1. **Feel the slope:** You use your feet to feel the ground. Which way slopes downward?\n2. **Take a step:** You take a step in that downward direction.\n3. **Repeat:** You feel the slope again, take another step downward, and repeat this process over and over until the ground feels completely flat. You have arrived at the bottom!\n\nThis is exactly how **Gradient Descent** works.\n\n---\n\n### How this applies to Computers (Machine Learning)\n\nIn machine learning, we want computers to make accurate predictions. \n\n* **The Mountain is "Error":** The top of the mountain represents making huge mistakes.

In [15]:
# Create a new thread id for this conversation
config = {"configurable": {"thread_id": '1234'}}

# ----- STEP 1: user asks a question -----
initial_input = {
    "messages": [
        ("user", "Explain gradient descent in very simple terms.")
    ]
}

# Invoke the graph for the first time
result = app.invoke(initial_input, config=config)

message

user_input=input(f"\nBackend message - {message} \n Approve this question?(y/n): ")

final_result=app.invoke(Command(resume={"approved":user_input}),
                        config=config)

final_result

{'messages': [HumanMessage(content='Explain gradient descent in very simple terms.', additional_kwargs={}, response_metadata={}, id='5b9c5443-edc8-4458-bbf6-eaf6d500dc0e'),
  AIMessage(content=[{'type': 'text', 'text': 'Imagine you are blindfolded and placed somewhere on a foggy mountain. Your goal is to find the very bottom of the valley. \n\nSince you can’t see, how do you get there?\n\n1. **Feel the slope:** You use your feet to feel the ground. Which way slopes downward?\n2. **Take a step:** You take a step in that downward direction.\n3. **Repeat:** You feel the slope again, take another step downward, and repeat this process over and over until the ground feels completely flat. You have arrived at the bottom!\n\nThis is exactly how **Gradient Descent** works.\n\n---\n\n### How this applies to Computers (Machine Learning)\n\nIn machine learning, we want computers to make accurate predictions. \n\n* **The Mountain is "Error":** The top of the mountain represents making huge mistake